# Tuning a random forest on the heart disease data
A random forest out of the box against three other classifiers, then tuning with `GridSearchCV` and
`RandomizedSearchCV`.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## 1. The data: 303 patients, 13 inputs, target 1 = heart disease

In [2]:
df = pd.read_csv("data/heart.csv")
print(df.shape)
df.head()

(303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [3]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(242, 13) (61, 13)


## 2. Four classifiers, out of the box: one test split, then 10-fold cross-validation

In [4]:
models = {
    "random forest": RandomForestClassifier(random_state=42),
    "gradient boosting": GradientBoostingClassifier(random_state=42),
    "SVM (SVC)": SVC(),
    "logistic regression": LogisticRegression(max_iter=5000),   # more iterations, so it converges
}
for name, model in models.items():
    test_acc = model.fit(X_train, y_train).score(X_test, y_test)
    cv_acc = cross_val_score(model, X, y, cv=10).mean()          # 10 splits, averaged
    print(f"{name:20s} test split {test_acc:.3f}   10-fold CV {cv_acc:.3f}")

random forest        test split 0.836   10-fold CV 0.832


gradient boosting    test split 0.770   10-fold CV 0.798
SVM (SVC)            test split 0.705   10-fold CV 0.660


logistic regression  test split 0.885   10-fold CV 0.818


In [5]:
# SVC measures distances, so it needs scaled inputs: with a scaler in a pipeline it catches up
svc_scaled = make_pipeline(StandardScaler(), SVC())
print("scaled SVC: test split", round(svc_scaled.fit(X_train, y_train).score(X_test, y_test), 3),
      " 10-fold CV", round(cross_val_score(svc_scaled, X, y, cv=10).mean(), 3))

scaled SVC: test split 0.869  10-fold CV 0.828


In [6]:
# a forest compares one column with a threshold, so scaling should change (almost) nothing: test it
rf_scaled = make_pipeline(StandardScaler(), RandomForestClassifier(random_state=42))
print("scaled forest: test split", round(rf_scaled.fit(X_train, y_train).score(X_test, y_test), 3),
      " 10-fold CV", round(cross_val_score(rf_scaled, X, y, cv=10).mean(), 3))

scaled forest: test split 0.836  10-fold CV 0.835


## 3. One hand-tuned setting: fewer rows per tree
One run of 10-fold cross-validation moves by about 0.01 when only the seed changes, so we use 500 trees (less forest randomness) and average 20 runs; each seed changes both the folds and the forest. Only `max_samples` changes between rows.

In [7]:
shares = [None, 0.75, 0.5, 0.3, 0.2]          # None: every tree gets a full-size bootstrap sample
cv_acc = {}
for share in shares:
    runs = [cross_val_score(RandomForestClassifier(n_estimators=500, max_samples=share, random_state=seed, n_jobs=-1), X, y,
                            cv=StratifiedKFold(10, shuffle=True, random_state=seed)).mean()
            for seed in range(20)]
    cv_acc[share] = np.array(runs)
    print(f"max_samples={str(share):5s} mean CV accuracy over 20 runs: {np.mean(runs):.3f}")

max_samples=None  mean CV accuracy over 20 runs: 0.826


max_samples=0.75  mean CV accuracy over 20 runs: 0.825


/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

...

/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

max_samples=0.2   mean CV accuracy over 20 runs: 0.834


In [8]:
# same seed, same folds: how often does 20% of the rows beat the full sample?
gain = cv_acc[0.2] - cv_acc[None]
print("mean gain:", round(gain.mean(), 3), "  runs won:", int((gain > 0).sum()), "of 20",
      "  run-to-run spread of one run (std):", round(cv_acc[None].std(), 3))

mean gain: 0.008   runs won: 16 of 20   run-to-run spread of one run (std): 0.009


In [9]:
# the reason (ESL 15.2): fewer rows per tree -> trees less alike. Measure it on the 61 test rows:
# average correlation between the 500 trees' predicted probabilities
def tree_correlation(share):
    forest = RandomForestClassifier(n_estimators=500, max_samples=share, random_state=0).fit(X_train, y_train)
    p = np.array([t.predict_proba(X_test.values)[:, 1] for t in forest.estimators_])
    c = np.corrcoef(p)
    return c[np.triu_indices_from(c, k=1)].mean()
for share in [None, 0.2]:
    print(f"max_samples={str(share):5s} average correlation between two trees: {tree_correlation(share):.2f}")

max_samples=None  average correlation between two trees: 0.44


max_samples=0.2   average correlation between two trees: 0.34


## 4. GridSearchCV: every combination of 4 hyperparameters

In [10]:
param_grid = {
    "n_estimators": [20, 60, 100, 120],   # number of trees
    "max_features": [0.2, 0.6, 1.0],      # share of columns considered at each split
    "max_depth": [2, 8, None],            # depth of each tree (None: fully grown)
    "max_samples": [0.5, 0.75, 1.0],      # share of rows per tree
}
print("combinations:", np.prod([len(v) for v in param_grid.values()]))

combinations: 108


In [11]:
start = time.time()
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, verbose=1, n_jobs=-1)
rf_grid.fit(X_train, y_train)
print(f"{time.time() - start:.1f} seconds")
print(rf_grid.best_params_)
print("best cross-validated accuracy:", round(rf_grid.best_score_, 3))
print("test accuracy of the best forest:", round(rf_grid.score(X_test, y_test), 3))

Fitting 5 folds for each of 108 candidates, totalling 540 fits


22.3 seconds
{'max_depth': 8, 'max_features': 0.2, 'max_samples': 0.75, 'n_estimators': 20}
best cross-validated accuracy: 0.843
test accuracy of the best forest: 0.869


In [12]:
# the five best combinations
res = pd.DataFrame(rf_grid.cv_results_)
cols = ["param_n_estimators", "param_max_features", "param_max_depth", "param_max_samples", "mean_test_score"]
res.sort_values("rank_test_score")[cols].head()

,param_n_estimators,param_max_features,param_max_depth,param_max_samples,mean_test_score
40,20,0.2,8,0.75,0.842942
76,20,0.2,None,0.75,0.838861
56,20,0.6,8,1.00,0.830612
44,20,0.2,8,1.00,0.830612
57,60,0.6,8,1.00,0.826531


## 4b. Did tuning help? Nested cross-validation
The grid's best score is the best of 108 scores on the same folds, so it is optimistic (selection bias). A fair test runs the whole grid search inside each outer fold and scores its winner on the outer fold, which the search never saw. 5-fold outer cross-validation, repeated 4 times: 20 outer folds.

In [13]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)
default = cross_validate(RandomForestClassifier(random_state=42, n_jobs=-1), X, y, cv=outer)["test_score"]
nested = cross_validate(GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, n_jobs=-1),
                        X, y, cv=outer, return_estimator=True)
tuned = nested["test_score"]
inner_best = np.mean([search.best_score_ for search in nested["estimator"]])
print("default forest, outer folds:      ", round(default.mean(), 3))
print("grid-tuned forest, outer folds:   ", round(tuned.mean(), 3))
print("grid's own best score (optimistic):", round(inner_best, 3))
print("tuned better on", int((tuned > default).sum()), "folds, equal on", int((tuned == default).sum()), "of", len(tuned))

default forest, outer folds:       0.819
grid-tuned forest, outer folds:    0.814
grid's own best score (optimistic): 0.845
tuned better on 5 folds, equal on 5 of 20


## 5. RandomizedSearchCV: a bigger grid, 10 random combinations
A first attempt mixes `bootstrap=False` with `max_samples`, which scikit-learn refuses: those candidates score NaN.

In [14]:
big_grid = dict(param_grid, bootstrap=[True, False], min_samples_split=[2, 5], min_samples_leaf=[1, 2])
print("combinations:", np.prod([len(v) for v in big_grid.values()]))
naive = RandomizedSearchCV(RandomForestClassifier(random_state=42), big_grid, cv=5, n_jobs=-1, random_state=42,
                           error_score=np.nan)
naive.fit(X_train, y_train)
print("candidates that failed:", int(np.isnan(naive.cv_results_["mean_test_score"]).sum()), "of 10")

combinations: 864


candidates that failed: 5 of 10


The fix: a list of two grids. `max_samples` appears only in the grid with `bootstrap=True`.

In [15]:
grids = [
    dict(param_grid, bootstrap=[True], min_samples_split=[2, 5], min_samples_leaf=[1, 2]),
    {"n_estimators": [20, 60, 100, 120], "max_features": [0.2, 0.6, 1.0], "max_depth": [2, 8, None],
     "bootstrap": [False], "min_samples_split": [2, 5], "min_samples_leaf": [1, 2]},
]
start = time.time()
rf_random = RandomizedSearchCV(RandomForestClassifier(random_state=42), grids, n_iter=10, cv=5, verbose=1,
                               n_jobs=-1, random_state=42)
rf_random.fit(X_train, y_train)
print(f"{time.time() - start:.1f} seconds")
print(rf_random.best_params_)
print("best cross-validated accuracy:", round(rf_random.best_score_, 3))
print("test accuracy of the best forest:", round(rf_random.score(X_test, y_test), 3))

Fitting 5 folds for each of 10 candidates, totalling 50 fits


1.8 seconds
{'n_estimators': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_samples': 1.0, 'max_features': 0.6, 'max_depth': None, 'bootstrap': True}
best cross-validated accuracy: 0.826
test accuracy of the best forest: 0.836
